# Final Master Pharmacy File

Builds the authoritative pharmacy dataset from SAPC registration records and Google Places geocode results.

Pipeline steps:
1. Filter SAPC raw data to Active records and deduplicate on Y-number
2. Attach geocoordinates from the SAPC geocode run
3. Clean and deduplicate the Places file
4. Stack SAPC and Places records; SAPC rows take priority in any collision
5. Resolve overlap between SAPC records lacking coordinates and Places-only rows
6. Re-geocode remaining no-coordinate records
7. Spatial validation: flag records outside expected province boundaries
8. Consolidate column names and export

Output: `PHARMACIES_MASTER_FINAL.csv`

## Imports and configuration

In [ ]:
import os, time, re
import pandas as pd
import numpy as np
import requests

BOX      = r'C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets'
BOX_SAPC = r'C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Source Files\SAPC'

sapc_raw    = pd.read_csv(f'{BOX_SAPC}\\sapc_raw.csv', dtype=str)
sapc_regeo  = pd.read_csv(f'{BOX}\\SAPC_places_results.csv', dtype=str)
places_full = pd.read_csv(f'{BOX}\\PHARMACIES_places_full_results.csv', dtype=str)

print(f'SAPC raw        : {len(sapc_raw):,}')
print(f'SAPC geocode    : {len(sapc_regeo):,}')
print(f'Places full     : {len(places_full):,}')

## Step 1 — Clean SAPC base

Filters to Active pharmacies only and deduplicates on Y-number.

In [ ]:
sapc = sapc_raw.copy()

if 'y_number' in sapc.columns:
    sapc = sapc.drop(columns=['y_number'])
sapc.rename(columns={'Account #': 'y_number'}, inplace=True)

sapc = sapc[sapc['Status'].str.strip().str.lower() == 'active'].copy()
sapc = sapc.drop_duplicates(subset=['y_number']).reset_index(drop=True)
sapc['source']        = 'sapc'
sapc['has_sapc_data'] = True

print(f'SAPC active deduped: {len(sapc):,}')
print(sapc['Province'].value_counts().to_string())

## Step 2 — Attach geocoordinates to SAPC records

In [ ]:
active_ynums = set(sapc['y_number'].dropna())

regeo_active = (
    sapc_regeo[
        (sapc_regeo['api_status'] == 'OK') &
        (sapc_regeo['sapc_y_number'].isin(active_ynums))
    ]
    .drop_duplicates(subset=['sapc_y_number'])
    [['sapc_y_number', 'place_id', 'matched_name', 'matched_address', 'lat', 'lng', 'types']]
    .rename(columns={'sapc_y_number': 'y_number'})
)

sapc = pd.merge(sapc, regeo_active, on='y_number', how='left')
print(f'SAPC geocode results: {len(regeo_active):,} / {len(sapc):,}')
print(f'  With coords   : {sapc["lat"].notna().sum():,}')
print(f'  Without coords: {sapc["lat"].isna().sum():,}')

## Step 3 — Clean Places file

In [ ]:
places_full['_ok'] = (places_full['api_status'] == 'OK').astype(int)
places = (
    places_full
    .sort_values('_ok', ascending=False)
    .drop_duplicates(subset=['place_id'])
    .drop(columns=['_ok'])
    .copy()
)
places['source']        = 'places'
places['has_sapc_data'] = False
print(f'Places deduped: {len(places):,}')

## Step 4 — Stack and deduplicate

SAPC rows take priority over Places rows in any collision on `place_id`.

In [ ]:
combined = pd.concat([sapc, places], ignore_index=True, sort=False)
combined['_sort'] = combined['source'].map({'sapc': 0, 'places': 1})
combined = combined.sort_values('_sort').reset_index(drop=True)

# Deduplicate records that have a place_id
has_place = combined['place_id'].notna() & (combined['place_id'].str.strip() != '')
no_place  = ~has_place

deduped_with_place = combined[has_place].drop_duplicates(subset=['place_id'], keep='first')

# For records without place_id, deduplicate on Y-number
no_place_df     = combined[no_place].copy()
has_y           = no_place_df['y_number'].notna() & (no_place_df['y_number'] != '')
deduped_with_y  = no_place_df[has_y].drop_duplicates(subset=['y_number'], keep='first')
deduped_no_y    = no_place_df[~has_y]

master = (
    pd.concat([deduped_with_place, deduped_with_y, deduped_no_y], ignore_index=True)
    .drop(columns=['_sort'], errors='ignore')
    .reset_index(drop=True)
)

print(f'After dedup: {len(master):,}')
print(master['source'].value_counts().to_string())

## Step 5 — Resolve SAPC/Places overlap

Identifies Places-only rows that appear to correspond to SAPC records missing coordinates, transfers the coordinates, and removes the now-redundant Places row.

In [ ]:
from rapidfuzz import fuzz

sapc_no_coords   = master[(master['source'] == 'sapc') & (master['lat'].isna())].copy()
places_only_rows = master[master['source'] == 'places'].copy()

print(f'SAPC rows without coords : {len(sapc_no_coords):,}')
print(f'Places-only rows         : {len(places_only_rows):,}')

def clean_for_match(s):
    if pd.isna(s): return ''
    s = str(s).lower()
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    for term in [r'\bpharmacy\b', r'\bpharm\b', r'\bchemist\b', r'\bapteek\b']:
        s = re.sub(term, '', s)
    return re.sub(r'\s+', ' ', s).strip()

sapc_no_coords['_clean']   = sapc_no_coords['Name'].apply(clean_for_match)
places_only_rows['_clean'] = places_only_rows['practice_name'].apply(clean_for_match)

COORD_COLS = ['place_id', 'matched_name', 'matched_address', 'lat', 'lng', 'types']
matches    = []

for p_idx, p_row in places_only_rows.iterrows():
    p_name = p_row['_clean']
    if not p_name:
        continue
    scores        = sapc_no_coords['_clean'].apply(lambda s: fuzz.token_sort_ratio(p_name, s))
    best_sapc_idx = scores.idxmax()
    best_score    = scores[best_sapc_idx]
    if best_score >= 85:
        matches.append({'places_idx': p_idx, 'sapc_idx': best_sapc_idx, 'score': best_score})

print(f'Matched pairs (score >= 85): {len(matches):,}')

places_idx_to_drop = set()
sapc_idx_updated   = set()

for m in matches:
    p_idx = m['places_idx']
    s_idx = m['sapc_idx']
    if s_idx in sapc_idx_updated:
        continue
    p_row = places_only_rows.loc[p_idx]
    for col in COORD_COLS:
        if col in master.columns:
            master.loc[s_idx, col] = p_row.get(col)
    places_idx_to_drop.add(p_idx)
    sapc_idx_updated.add(s_idx)

print(f'SAPC rows updated with coords: {len(sapc_idx_updated):,}')
print(f'Places rows removed           : {len(places_idx_to_drop):,}')

master = master[~master.index.isin(places_idx_to_drop)].reset_index(drop=True)
master.drop(columns=['_clean'], inplace=True, errors='ignore')

## Spatial validation helpers

Defines province bounding boxes (with a 5-mile buffer) and a spatial flag function used throughout the remaining steps.

In [ ]:
BUFFER = 0.072   # ~5 miles in degrees

BOUNDS = {
    'gauteng':       {'lat_min': -26.7-BUFFER, 'lat_max': -25.2+BUFFER,
                      'lng_min':  27.5-BUFFER, 'lng_max':  29.1+BUFFER},
    'kwazulu-natal': {'lat_min': -31.1-BUFFER, 'lat_max': -26.8+BUFFER,
                      'lng_min':  28.8-BUFFER, 'lng_max':  32.9+BUFFER},
}
SA_BOUNDS = {'lat_min': -35.0, 'lat_max': -22.0,
             'lng_min':  16.0, 'lng_max':  33.5}


def prov_norm(s):
    s = str(s).strip().lower() if pd.notna(s) else ''
    if 'kwazulu' in s or 'kzn' in s or 'natal' in s: return 'kwazulu-natal'
    if 'gauteng' in s: return 'gauteng'
    return s


def spatial_flag(row):
    lat  = pd.to_numeric(row.get('lat'), errors='coerce')
    lng  = pd.to_numeric(row.get('lng'), errors='coerce')
    prov = prov_norm(row.get('Province') or row.get('province') or '')
    if pd.isna(lat) or pd.isna(lng):
        return 'no_coords'
    if not (SA_BOUNDS['lat_min'] <= lat <= SA_BOUNDS['lat_max'] and
            SA_BOUNDS['lng_min'] <= lng <= SA_BOUNDS['lng_max']):
        return 'outside_sa'
    if prov in BOUNDS:
        b = BOUNDS[prov]
        if b['lat_min'] <= lat <= b['lat_max'] and b['lng_min'] <= lng <= b['lng_max']:
            return 'ok'
        return 'outside_province'
    return 'ok'


master['spatial_check'] = master.apply(spatial_flag, axis=1)
print('Spatial check:')
print(master['spatial_check'].value_counts().to_string())

## Step 6 — Re-geocode records with no coordinates

Attempts to resolve the remaining no-coordinate rows via a second Places API pass. Only applies results that fall within South Africa.

In [ ]:
API_KEY = os.getenv('GOOGLE_MAPS_API_KEY')
REGEO2_CHECKPOINT = 'regeocode2_checkpoint.csv'

no_coords = master[master['spatial_check'] == 'no_coords'].copy()
print(f'Rows to re-geocode: {len(no_coords):,}')
print(no_coords['source'].value_counts().to_string())


def build_query(row):
    name = (str(row.get('pharmacy_name') or row.get('Name') or
                row.get('practice_name') or '')).strip()
    city = (str(row.get('sapc_city') or row.get('city') or '')).strip()
    prov = (str(row.get('Province') or row.get('province') or '')).strip()
    parts = [p for p in [name, city, prov, 'South Africa'] if p and p != 'nan']
    return ', '.join(parts)


def search_place_text(query):
    url    = 'https://maps.googleapis.com/maps/api/place/textsearch/json'
    params = {'query': query, 'type': 'pharmacy', 'key': API_KEY}
    r      = requests.get(url, params=params, timeout=30)
    try:
        data = r.json()
    except Exception:
        return {'api_status': 'BAD_JSON', 'place_id': None,
                'matched_name': None, 'matched_address': None, 'lat': None, 'lng': None}
    status = data.get('status')
    if status != 'OK':
        return {'api_status': status, 'place_id': None,
                'matched_name': None, 'matched_address': None, 'lat': None, 'lng': None}
    result = data['results'][0]
    return {
        'api_status':      status,
        'place_id':        result.get('place_id'),
        'matched_name':    result.get('name'),
        'matched_address': result.get('formatted_address'),
        'lat':             result.get('geometry', {}).get('location', {}).get('lat'),
        'lng':             result.get('geometry', {}).get('location', {}).get('lng'),
    }


no_coords['query_string'] = no_coords.apply(build_query, axis=1)

if os.path.exists(REGEO2_CHECKPOINT):
    existing  = pd.read_csv(REGEO2_CHECKPOINT, dtype=str)
    done_idx  = set(existing['master_idx'].dropna().astype(str))
    remaining = no_coords[~no_coords.index.astype(str).isin(done_idx)].copy()
    print(f'Checkpoint: {len(existing):,} done | Remaining: {len(remaining):,}')
else:
    existing  = pd.DataFrame()
    remaining = no_coords.copy()
    print(f'No checkpoint — querying all {len(remaining):,}')

results = []
for n, (idx, row) in enumerate(remaining.iterrows(), start=1):
    result = search_place_text(row['query_string'])
    results.append({'master_idx': idx, 'y_number': row.get('y_number'),
                    'query_string': row['query_string'], **result})
    if n % 100 == 0:
        print(f'  {n:,} / {len(remaining):,}')
    if n % 100 == 0:
        chk = pd.concat([existing, pd.DataFrame(results)], ignore_index=True)
        chk.drop_duplicates(subset=['master_idx'], keep='first', inplace=True)
        chk.to_csv(REGEO2_CHECKPOINT, index=False)
    time.sleep(0.2)

combined_regeo = pd.concat([existing, pd.DataFrame(results)], ignore_index=True)
combined_regeo.drop_duplicates(subset=['master_idx'], keep='first', inplace=True)
combined_regeo.to_csv(REGEO2_CHECKPOINT, index=False)
print('API status:')
print(combined_regeo['api_status'].value_counts().to_string())

# Apply results that fall within South Africa bounds
UPDATE_COLS = ['place_id', 'matched_name', 'matched_address', 'lat', 'lng']
ok_results  = combined_regeo[combined_regeo['api_status'] == 'OK'].copy()
ok_results['master_idx'] = ok_results['master_idx'].astype(int)

for _, row in ok_results.iterrows():
    idx = row['master_idx']
    lat = pd.to_numeric(row.get('lat'), errors='coerce')
    lng = pd.to_numeric(row.get('lng'), errors='coerce')
    if pd.isna(lat) or pd.isna(lng):
        continue
    if not (SA_BOUNDS['lat_min'] <= lat <= SA_BOUNDS['lat_max'] and
            SA_BOUNDS['lng_min'] <= lng <= SA_BOUNDS['lng_max']):
        continue
    for col in UPDATE_COLS:
        if col in master.columns:
            master.loc[idx, col] = row.get(col)

master['spatial_check'] = master.apply(spatial_flag, axis=1)
print('\nSpatial check after re-geocode:')
print(master['spatial_check'].value_counts().to_string())

## QA — Inspect flagged records

Reviews records with coordinate issues before the final clean. Records outside province boundaries are often caused by a same-name pharmacy in a different province receiving the geocode match.

In [ ]:
outside_sa   = master[master['spatial_check'] == 'outside_sa'].copy()
outside_prov = master[master['spatial_check'] == 'outside_province'].copy()

print(f'Outside South Africa: {len(outside_sa)}')
print(f'Outside province    : {len(outside_prov)}')

disp_cols = ['y_number', 'Name', 'practice_name', 'Province', 'province', 'lat', 'lng', 'matched_address', 'source']

if len(outside_sa):
    display(outside_sa[[c for c in disp_cols if c in outside_sa.columns]].head(20))

if len(outside_prov):
    display(outside_prov[[c for c in disp_cols if c in outside_prov.columns]].head(30))

# Duplicate place_ids
has_place   = master['place_id'].notna() & (master['place_id'].str.strip() != '')
place_dupes = master[has_place & master.duplicated(subset=['place_id'], keep=False)]
print(f'\nDuplicate place_ids: {len(place_dupes)} rows ({len(place_dupes)//2} pairs)')

# Duplicate Y-numbers
has_y   = master['y_number'].notna() & (master['y_number'].str.strip() != '')
y_dupes = master[has_y & master.duplicated(subset=['y_number'], keep=False)]
print(f'Duplicate Y-numbers: {len(y_dupes)} rows ({len(y_dupes)//2} pairs)')

## Final deduplication and spatial filter

Removes records outside South Africa or misassigned to the wrong province, then performs a final place_id dedup.

In [ ]:
before = len(master)

# Final place_id dedup with SAPC priority
master['_sort'] = master['source'].map({'sapc': 0, 'places': 1}).fillna(2)
master = (
    master.sort_values('_sort')
    .drop_duplicates(subset=['place_id'], keep='first')
    .drop(columns=['_sort'])
    .reset_index(drop=True)
)
print(f'After place_id dedup: {len(master):,} (removed {before - len(master):,})')

# Remove spatially invalid records
remove_flags = ['outside_sa', 'outside_province']
flagged      = master[master['spatial_check'].isin(remove_flags)]
master       = master[~master['spatial_check'].isin(remove_flags)].reset_index(drop=True)

print(f'Removed outside_sa       : {(flagged["spatial_check"]=="outside_sa").sum():,}')
print(f'Removed outside_province : {(flagged["spatial_check"]=="outside_province").sum():,}')
print(f'\nClean master: {len(master):,} rows')
print(master['source'].value_counts().to_string())
print()
print(master['spatial_check'].value_counts().to_string())

## Consolidate columns and export

Generates a unique `record_id`, consolidates name/address/province from the best available source, and selects the final column set.

In [ ]:
master = master.reset_index(drop=True)
master['record_id'] = ['PH' + str(i).zfill(5) for i in range(1, len(master) + 1)]

# Use best available name/address/province fields
master['name']     = master['pharmacy_name'].fillna(master['Name']).fillna(master.get('practice_name', np.nan))
master['address']  = master['street_address'].fillna(master['matched_address'])
master['province'] = master['Province'].fillna(master.get('province', np.nan))
master['status']   = master.get('status', pd.Series(dtype=str)).fillna(master['Status'])
master['inspection'] = master.get('inspection', pd.Series(dtype=str)).fillna(master['Inspection'])

# Consolidate opening hours into a single column if hour columns exist
hour_cols = [c for c in master.columns
             if c in ['mondays_to_thursdays', 'fridays', 'saturdays',
                      'sundays', 'public_holidays', 'wednesdays']]
if hour_cols:
    master['opening_hours'] = master[hour_cols].apply(
        lambda row: ' | '.join(
            f'{col}: {val}' for col, val in row.items()
            if pd.notna(val) and str(val).strip() not in ('', 'nan')
        ), axis=1
    ).replace('', np.nan)

FINAL_COLS = [
    'record_id', 'y_number', 'pharmacy_id', 'place_id',
    'name', 'category', 'status', 'licence_number', 'registration_date',
    'owner', 'inspection',
    'address', 'city', 'province', 'telephone',
    'matched_name', 'matched_address', 'lat', 'lng', 'types',
    'opening_hours',
    'source', 'spatial_check',
]

final_cols   = [c for c in FINAL_COLS if c in master.columns]
master_clean = master[final_cols].copy()

print(f'Final master: {len(master_clean):,} rows x {len(master_clean.columns)} columns')
print()
print('Column completeness:')
for col in master_clean.columns:
    n   = master_clean[col].notna().sum()
    pct = n / len(master_clean) * 100
    print(f'  {col:30s} {n:>5,} / {len(master_clean):,} ({pct:4.0f}%)')

display(master_clean.head(5))

In [ ]:
OUTPUT_PATH = f'{BOX}\\PHARMACIES_MASTER_FINAL.csv'
master_clean.to_csv(OUTPUT_PATH, index=False)

print(f'{len(master_clean):,} pharmacies saved to {OUTPUT_PATH}')
print()
print('Province breakdown:')
print(master_clean['province'].value_counts().to_string())
print()
print('Status breakdown:')
print(master_clean['status'].value_counts(dropna=False).to_string())
print()
print('Source breakdown:')
print(master_clean['source'].value_counts().to_string())
print()
print(f'With coordinates : {master_clean["lat"].notna().sum():,}')
print(f'Without coords   : {master_clean["lat"].isna().sum():,}')